# Random Survival Forests  

Goal of this notebook is to: 
- Train a RSF
- Evaluate predictions for RSF  
- Use Permutation Importance to determine which features are the most "predictive" of performance   

Some of the code for this notebook uses data and code from `scikit-survival` Random Survival Forests [Tutorial](https://scikit-survival.readthedocs.io/en/stable/user_guide/random-survival-forest.html). 

## 0. Preliminaries

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

from sklearn import set_config
from sklearn.model_selection import train_test_split

from sksurv.datasets import load_breast_cancer
from sksurv.ensemble import RandomSurvivalForest
from sksurv.preprocessing import OneHotEncoder

set_config(display="text")  # displays text representation of estimators

In [2]:
X, y = load_breast_cancer()
Xt = OneHotEncoder().fit_transform(X)

## 1. Fitting and Evaluating a RSF

In [3]:
seed = 123

# split into Train & Test (for illustrative purposes, we are not doing Cross-Validation yet)
X_train, X_test, y_train, y_test = train_test_split(Xt, y, test_size=0.10, random_state=seed)

rsf = RandomSurvivalForest(
    n_estimators=1000, min_samples_split=10, min_samples_leaf=15, n_jobs=-1, random_state=seed
)
rsf.fit(X_train, y_train)

RandomSurvivalForest(min_samples_leaf=15, min_samples_split=10,
                     n_estimators=1000, n_jobs=-1, random_state=123)

We evaluate to see how well the model does on the test data

In [4]:
c_index = rsf.score(X_test, y_test)
f"{c_index:.5f}"

'0.48936'

## 2. Permutation-Based Feature Importance

To determine the feature importance of each covariate, RSF relies on a measure of *impurity* for child nodes and importance is defined as the decrease in impurity due to a split. 

With *permutation* in the name, the number of times you permute a feature is `n_repeats`. The greater `n_repeats` is, the more computationally expensive the `permutation_importance` function becomes (which means more waiting time for you). In many cases, it may be favorable to use SHAPley values which is a game-theoretic approach to determining feature importance.

In [7]:
from sklearn.inspection import permutation_importance

result = permutation_importance(rsf, X_test, y_test, n_repeats=5, random_state=seed) # with n_repeats = 5, this takes ~ 20 seconds to run
pd.DataFrame(
    {
        k: result[k]
        for k in (
            "importances_mean",
            "importances_std",
        )
    },
    index=X_test.columns,
).sort_values(by="importances_mean", ascending=False)

,importances_mean,importances_std
X218883_s_at,0.068085,0.056132
size,0.017021,0.008511
X221882_s_at,0.012766,0.010423
X204015_s_at,0.008511,0.031844
X202240_at,0.008511,0.079154
...,...,...
X205034_at,-0.012766,0.025532
X214806_at,-0.012766,0.025532
X208180_s_at,-0.012766,0.025532
X201664_at,-0.017021,0.008511
